# 🌍 Climate & Health Risk Prediction - Advanced Model
## Ensemble Approach with Advanced Feature Engineering

**Objective:** Maximize the competition metric: **Final Score = 0.60 × F1 + 0.40 × ROC-AUC**

### Key Improvements over Baseline:
1. **Advanced Feature Engineering** - 50+ engineered features
2. **Multiple Powerful Models** - XGBoost, LightGBM, CatBoost, Random Forest
3. **Stratified Cross-Validation** - Robust performance estimation
4. **Threshold Optimization** - Find optimal classification threshold
5. **Ensemble Predictions** - Weighted combination of best models

## 📚 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score, classification_report

# Advanced models
try:
    import xgboost as xgb
    HAS_XGB = True
    print("✓ XGBoost loaded")
except ImportError:
    HAS_XGB = False
    print("✗ XGBoost not available")

try:
    import lightgbm as lgb
    HAS_LGBM = True
    print("✓ LightGBM loaded")
except ImportError:
    HAS_LGBM = False
    print("✗ LightGBM not available")

try:
    import catboost as cb
    HAS_CAT = True
    print("✓ CatBoost loaded")
except ImportError:
    HAS_CAT = False
    print("✗ CatBoost not available")

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

## ⚙️ Configuration

In [ ]:
# Configuration
RANDOM_STATE = 42
N_SPLITS = 5
TARGET = "is_climate_sensitive"
ID_COL = "ID"

print(f"Random State: {RANDOM_STATE}")
print(f"Cross-Validation Folds: {N_SPLITS}")
print(f"Target Column: {TARGET}")

## 📥 Load and Merge Data

In [ ]:
# Load datasets
print("=" * 60)
print("LOADING DATA")
print("=" * 60)

train = pd.read_csv("Train.csv")
test = pd.read_csv("Test.csv")
climate_features = pd.read_csv("climate_features.csv")

# Drop deathdate from climate features to avoid duplicates
climate_features = climate_features.drop(columns=["deathdate"], errors='ignore')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"Climate features shape: {climate_features.shape}")

# Merge datasets
train = train.merge(climate_features, on=ID_COL, how="left")
test = test.merge(climate_features, on=ID_COL, how="left")

print(f"\nMerged Train: {train.shape}")
print(f"Merged Test: {test.shape}")

# Quick look at target distribution
print(f"\nTarget Distribution:")
print(train[TARGET].value_counts(normalize=True).round(3))

## 🔧 Advanced Feature Engineering

In [ ]:
def engineer_features(df, is_train=True):
    """Advanced feature engineering for climate-health prediction"""
    df = df.copy()
    
    # Convert date
    df["deathdate"] = pd.to_datetime(df["deathdate"], errors="coerce")
    
    # ===== TEMPORAL/CYCLICAL FEATURES =====
    df["day_of_year"] = df["deathdate"].dt.dayofyear
    df["month"] = df["deathdate"].dt.month
    df["season"] = ((df["month"] % 12 + 3) // 3) % 4  # 0=DJF, 1=MAM, 2=JJA, 3=SON
    
    # Cyclical encoding (captures seasonal patterns)
    df["day_sin"] = np.sin(2 * np.pi * df["day_of_year"] / 365)
    df["day_cos"] = np.cos(2 * np.pi * df["day_of_year"] / 365)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    
    # ===== AGE-BASED RISK GROUPS =====
    # Use np.digitize to avoid NaN issues with pd.cut
    df["age_group"] = np.digitize(df["age"], bins=[5, 18, 35, 50, 65])
    df["is_vulnerable_age"] = ((df["age"] < 5) | (df["age"] > 65)).astype(int)
    df["is_elderly"] = (df["age"] > 60).astype(int)
    df["is_child"] = (df["age"] < 5).astype(int)
    df["age_squared"] = df["age"] ** 2
    
    # ===== TEMPERATURE FEATURES =====
    df["temperature_range"] = df["max_temperature"] - df["min_temperature"]
    df["temp_extreme_high"] = (df["avg_temperature"] > 25).astype(int)
    df["temp_extreme_low"] = (df["avg_temperature"] < 18).astype(int)
    df["temp_anomaly"] = df["avg_temperature"] - df["tavg_30d"]
    df["temp_anomaly_7d"] = df["avg_temperature"] - df["tavg_7d"]
    df["max_temp_extreme"] = (df["max_temperature"] > 30).astype(int)
    df["min_temp_extreme"] = (df["min_temperature"] < 15).astype(int)
    df["temp_variability"] = df["temp_range_mean_30d"] / (df["avg_temperature"] + 1)
    
    # ===== PRECIPITATION FEATURES =====
    df["is_heavy_rain"] = (df["precipitation"] > 5).astype(int)
    df["is_rainy_day"] = (df["precipitation"] > 0).astype(int)
    df["rain_intensity"] = df["precipitation"] / (df["rain_days_30d"] + 1)
    df["rain_anomaly"] = df["rain_sum_7d"] - (df["rain_sum_30d"] / 30 * 7)
    df["recent_rain_ratio"] = df["rain_sum_7d"] / (df["rain_sum_30d"] + 1)
    df["prolonged_rain"] = (df["rain_days_30d"] > 20).astype(int)
    
    # ===== NDVI (VEGETATION) FEATURES =====
    df["ndvi_change"] = df["ndvi_30d"] - df["ndvi_90d"]
    df["ndvi_high"] = (df["ndvi_30d"] > 0.7).astype(int)
    df["ndvi_low"] = (df["ndvi_30d"] < 0.4).astype(int)
    df["vegetation_stress"] = ((df["ndvi_30d"] < 0.5) & (df["avg_temperature"] > 23)).astype(int)
    
    # ===== ELEVATION/TERRAIN FEATURES =====
    df["high_elevation"] = (df["elevation"] > 1500).astype(int)
    df["steep_slope"] = (df["slope"] > 5).astype(int)
    df["elevation_temp_interaction"] = df["elevation"] * df["avg_temperature"] / 1000
    
    # ===== HOT/COLD DAY FEATURES =====
    df["heat_wave"] = (df["hot_days_30d"] > 5).astype(int)
    df["extreme_heat"] = (df["hot_days_30d"] > 10).astype(int)
    
    # ===== INTERACTION FEATURES =====
    # Age interactions with climate
    df["elderly_heat"] = df["is_elderly"] * df["temp_extreme_high"]
    df["child_rain"] = df["is_child"] * df["is_rainy_day"]
    df["vulnerable_extreme"] = df["is_vulnerable_age"] * df["temp_extreme_high"]
    
    # Rain and temperature interaction
    df["hot_humid"] = df["temp_extreme_high"] * df["is_rainy_day"]
    df["cold_wet"] = df["temp_extreme_low"] * df["is_heavy_rain"]
    
    # Elevation interactions
    df["high_elev_cold"] = df["high_elevation"] * df["temp_extreme_low"]
    df["slope_rain"] = df["steep_slope"] * df["is_heavy_rain"]
    
    # NDVI interactions
    df["hot_dry"] = df["temp_extreme_high"] * df["ndvi_low"]
    df["vegetation_heat"] = df["ndvi_high"] * df["temp_extreme_high"]
    
    # ===== COMPOSITE RISK SCORES =====
    df["climate_stress_score"] = (
        df["hot_days_30d"] / 10 +
        df["temp_anomaly"].abs() +
        df["rain_anomaly"].abs() / 10 +
        df["is_vulnerable_age"] * 2
    )
    
    df["environmental_risk"] = (
        df["temp_extreme_high"].astype(int) +
        df["is_heavy_rain"].astype(int) +
        df["ndvi_low"].astype(int) +
        df["high_elevation"].astype(int)
    )
    
    # ===== LOCATION-BASED FEATURES =====
    df["lat_bin"] = np.digitize(df["latitude"], bins=[0.5, 1.0])
    df["lon_bin"] = np.digitize(df["longitude"], bins=[33, 34])
    df["dist_from_equator"] = df["latitude"].abs()
    
    # ===== GENDER/ZONE ENCODING =====
    df["is_male"] = (df["gender"] == "Male").astype(int)
    df["is_female"] = (df["gender"] == "Female").astype(int)
    df["is_rural"] = (df["zone"] == "Rural").astype(int)
    df["is_urban"] = (df["zone"].isin(["Urban", "Peri_urban"])).astype(int)
    
    # Drop raw date
    df = df.drop(columns=["deathdate"], errors='ignore')
    
    return df

# Apply feature engineering
print("=" * 60)
print("FEATURE ENGINEERING")
print("=" * 60)

train = engineer_features(train, is_train=True)
test = engineer_features(test, is_train=False)

print(f"Features after engineering: {train.shape[1] - 2}")
print(f"New features created: ~{train.shape[1] - 29}")

## 📊 Prepare Features for Modeling

In [ ]:
# Identify feature columns (exclude non-feature columns)
# Note: zone and gender ARE features we want to use - they get one-hot encoded
exclude_cols = [ID_COL, TARGET, "location", "age_group", "lat_bin", "lon_bin"]
feature_cols = [c for c in train.columns if c not in exclude_cols]

# Separate features and target
X = train[feature_cols].copy()
y = train[TARGET].copy()
X_test = test[feature_cols].copy()

# Identify categorical and numeric columns from the feature set
cat_cols = ['zone', 'gender']  # Categoricals to one-hot encode
num_cols = [c for c in feature_cols if c not in cat_cols]

print(f"Numeric features: {len(num_cols)}")
print(f"Categorical features: {len(cat_cols)}")
print(f"Total features: {len(feature_cols)}")

# Verify categorical columns exist in X
for col in cat_cols:
    if col not in X.columns:
        print(f"WARNING: {col} not in X.columns!")
        
# Check for any missing values
missing = X.isnull().sum().sum()
print(f"\nTotal missing values in X: {missing}")

## 🔧 Preprocessing Pipeline

In [ ]:
# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), cat_cols),
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols)
    ]
)

print("Preprocessor configured:")
print(f"  - Categorical: {cat_cols} -> OneHotEncoding")
print(f"  - Numeric: {len(num_cols)} features -> Median Imputation + StandardScaler")

## 🤖 Define Models

In [ ]:
def get_models():
    """Return dictionary of models to evaluate"""
    models = {}
    
    # Logistic Regression (interpretable baseline)
    models['LogisticRegression'] = Pipeline([
        ("preprocess", preprocessor),
        ("model", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            C=0.5,
            solver="lbfgs",
            random_state=RANDOM_STATE
        ))
    ])
    
    # Random Forest (robust ensemble)
    models['RandomForest'] = Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_depth=12,
            min_samples_split=5,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1
        ))
    ])
    
    # Gradient Boosting
    models['GradientBoosting'] = Pipeline([
        ("preprocess", preprocessor),
        ("model", GradientBoostingClassifier(
            n_estimators=150,
            max_depth=6,
            learning_rate=0.05,
            min_samples_split=10,
            min_samples_leaf=5,
            random_state=RANDOM_STATE
        ))
    ])
    
    # XGBoost (if available)
    if HAS_XGB:
        models['XGBoost'] = Pipeline([
            ("preprocess", preprocessor),
            ("model", xgb.XGBClassifier(
                n_estimators=200,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                scale_pos_weight=1.5,  # Handle class imbalance
                random_state=RANDOM_STATE,
                eval_metric='logloss',
                use_label_encoder=False
            ))
        ])
    
    # LightGBM (if available)
    if HAS_LGBM:
        models['LightGBM'] = Pipeline([
            ("preprocess", preprocessor),
            ("model", lgb.LGBMClassifier(
                n_estimators=200,
                max_depth=8,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                verbose=-1
            ))
        ])
    
    # CatBoost (if available)
    if HAS_CAT:
        models['CatBoost'] = Pipeline([
            ("preprocess", preprocessor),
            ("model", cb.CatBoostClassifier(
                iterations=200,
                depth=6,
                learning_rate=0.05,
                class_weights="balanced",
                random_state=RANDOM_STATE,
                verbose=0
            ))
        ])
    
    return models

models = get_models()
print(f"\nModels configured: {list(models.keys())}")

## 📈 Cross-Validation Evaluation

In [ ]:
print("=" * 60)
print("CROSS-VALIDATION EVALUATION")
print("=" * 60)

cv_results = {}
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

for name, model in models.items():
    print(f"\nEvaluating {name}...")
    
    # Get probability predictions via cross-validation
    cv_proba = cross_val_predict(model, X, y, cv=skf, method="predict_proba")[:, 1]
    cv_pred = (cv_proba >= 0.5).astype(int)
    
    # Calculate metrics
    f1 = f1_score(y, cv_pred)
    auc = roc_auc_score(y, cv_proba)
    final_score = 0.60 * f1 + 0.40 * auc
    
    cv_results[name] = {
        'model': model,
        'f1': f1,
        'auc': auc,
        'final_score': final_score,
        'cv_proba': cv_proba
    }
    
    print(f"  F1 Score:    {f1:.4f}")
    print(f"  ROC-AUC:     {auc:.4f}")
    print(f"  Final Score: {final_score:.4f}")

# Display results table
print("\n" + "=" * 60)
print("CV RESULTS SUMMARY")
print("=" * 60)

results_df = pd.DataFrame({
    'Model': list(cv_results.keys()),
    'F1 Score': [cv_results[m]['f1'] for m in cv_results],
    'ROC-AUC': [cv_results[m]['auc'] for m in cv_results],
    'Final Score': [cv_results[m]['final_score'] for m in cv_results]
}).sort_values('Final Score', ascending=False)

print(results_df.to_string(index=False))

## 🎯 Threshold Optimization

In [ ]:
print("=" * 60)
print("THRESHOLD OPTIMIZATION")
print("=" * 60)

# Find best model
best_model_name = max(cv_results, key=lambda x: cv_results[x]['final_score'])
print(f"Best model: {best_model_name}")

# Optimize threshold for best model
best_proba = cv_results[best_model_name]['cv_proba']

thresholds = np.arange(0.30, 0.70, 0.01)
best_threshold = 0.5
best_combined = 0
best_f1_at_thresh = 0

threshold_results = []
for thresh in thresholds:
    pred = (best_proba >= thresh).astype(int)
    f1_t = f1_score(y, pred)
    auc_t = roc_auc_score(y, best_proba)  # AUC doesn't depend on threshold
    combined = 0.60 * f1_t + 0.40 * auc_t
    threshold_results.append({'threshold': thresh, 'f1': f1_t, 'auc': auc_t, 'combined': combined})
    
    if combined > best_combined:
        best_combined = combined
        best_threshold = thresh
        best_f1_at_thresh = f1_t

print(f"\nOptimal threshold: {best_threshold:.2f}")
print(f"F1 at optimal: {best_f1_at_thresh:.4f}")
print(f"Best combined score: {best_combined:.4f}")

# Plot threshold vs score
thresh_df = pd.DataFrame(threshold_results)
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(thresh_df['threshold'], thresh_df['f1'], 'b-', label='F1 Score', linewidth=2)
plt.axvline(best_threshold, color='r', linestyle='--', label=f'Optimal: {best_threshold:.2f}')
plt.xlabel('Threshold')
plt.ylabel('F1 Score')
plt.title('F1 Score vs Threshold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(thresh_df['threshold'], thresh_df['combined'], 'g-', label='Combined Score', linewidth=2)
plt.axvline(best_threshold, color='r', linestyle='--', label=f'Optimal: {best_threshold:.2f}')
plt.xlabel('Threshold')
plt.ylabel('Combined Score (0.6×F1 + 0.4×AUC)')
plt.title('Combined Score vs Threshold')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 🏆 Ensemble Predictions

In [ ]:
print("=" * 60)
print("ENSEMBLE PREDICTIONS")
print("=" * 60)

# Calculate model weights based on CV performance
total_score = sum(cv_results[m]['final_score'] for m in cv_results)
model_weights = {m: cv_results[m]['final_score'] / total_score for m in cv_results}

print("Model weights for weighted ensemble:")
for m, w in sorted(model_weights.items(), key=lambda x: -x[1]):
    print(f"  {m:20s}: {w:.3f}")

# Train all models and create weighted ensemble
ensemble_proba = np.zeros(len(X_test))

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X, y)
    test_proba = model.predict_proba(X_test)[:, 1]
    ensemble_proba += model_weights[name] * test_proba

# Also compute simple average ensemble
all_test_probas = []
for name, model in models.items():
    test_proba = model.predict_proba(X_test)[:, 1]
    all_test_probas.append(test_proba)

simple_avg = np.mean(all_test_probas, axis=0)

# Blend weighted ensemble with simple average
final_proba = 0.7 * ensemble_proba + 0.3 * simple_avg

# Apply optimal threshold
final_pred = (final_proba >= best_threshold).astype(int)

print(f"\nEnsemble probability range: [{final_proba.min():.3f}, {final_proba.max():.3f}]")
print(f"Predicted positives: {final_pred.sum()} / {len(final_pred)} ({100*final_pred.mean():.1f}%)")

## 📊 Feature Importance Analysis

In [ ]:
print("=" * 60)
print("FEATURE IMPORTANCE")
print("=" * 60)

# Use the best model for feature importance
best_model = models[best_model_name]
best_model.fit(X, y)

try:
    # Get the trained model from pipeline
    trained_model = best_model.named_steps['model']
    
    if hasattr(trained_model, 'feature_importances_'):
        importances = trained_model.feature_importances_
        
        # Get feature names after preprocessing
        ohe = best_model.named_steps['preprocess'].named_transformers_['cat'].named_steps['onehot']
        cat_feature_names = ohe.get_feature_names_out(cat_cols).tolist()
        feature_names = cat_feature_names + num_cols
        
        # Create importance dataframe
        imp_df = pd.DataFrame({
            'feature': feature_names,
            'importance': importances
        }).sort_values('importance', ascending=False)
        
        # Plot top 25 features
        plt.figure(figsize=(10, 8))
        top_features = imp_df.head(25)
        plt.barh(range(len(top_features)), top_features['importance'].values)
        plt.yticks(range(len(top_features)), top_features['feature'].values)
        plt.xlabel('Importance')
        plt.title(f'Top 25 Feature Importances ({best_model_name})')
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()
        
        print("\nTop 20 features by importance:")
        for i, row in imp_df.head(20).iterrows():
            print(f"  {row['feature']:35s}: {row['importance']:.4f}")
    else:
        print("Model does not have feature_importances_ attribute")
except Exception as e:
    print(f"Could not extract feature importance: {e}")

## 📝 Create Submission File

In [ ]:
print("=" * 60)
print("CREATING SUBMISSION")
print("=" * 60)

submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    "TargetF1": final_pred,
    "TargetRAUC": final_proba
})

submission.to_csv("advanced_submission.csv", index=False)

print(f"Submission saved: advanced_submission.csv")
print(f"Shape: {submission.shape}")
print(f"\nTargetF1 distribution:")
print(submission['TargetF1'].value_counts())
print(f"\nTargetRAUC statistics:")
print(submission['TargetRAUC'].describe())

## 📋 Final Summary

In [ ]:
print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)

print("\n📊 Cross-Validation Results:")
print("-" * 60)
for name in results_df['Model']:
    r = cv_results[name]
    print(f"{name:20s} | F1: {r['f1']:.4f} | AUC: {r['auc']:.4f} | Score: {r['final_score']:.4f}")

print("\n" + "=" * 60)
print("🏆 BEST MODEL CONFIGURATION")
print("=" * 60)
print(f"Best Model:          {best_model_name}")
print(f"Optimal Threshold:   {best_threshold:.2f}")
print(f"CV F1 Score:         {cv_results[best_model_name]['f1']:.4f}")
print(f"CV ROC-AUC:          {cv_results[best_model_name]['auc']:.4f}")
print(f"CV Final Score:      {cv_results[best_model_name]['final_score']:.4f}")
print(f"\n📁 Submission file:    advanced_submission.csv")
print(f"📊 Predicted positives: {final_pred.sum()} ({100*final_pred.mean():.1f}%)")

print("\n" + "=" * 60)
print("✅ KEY IMPROVEMENTS OVER BASELINE")
print("=" * 60)
print("""
1. Advanced Feature Engineering:
   - Cyclical time features (day_sin, day_cos, month_sin, month_cos)
   - Age-based risk groups and vulnerability indicators
   - Temperature anomaly and variability features
   - Precipitation intensity and anomaly features
   - NDVI vegetation stress indicators
   - Interaction terms (age×climate, elevation×temperature, etc.)
   - Composite risk scores

2. Multiple Powerful Models:
   - XGBoost, LightGBM, CatBoost (gradient boosting)
   - Random Forest (bagging ensemble)
   - Gradient Boosting Classifier
   - Logistic Regression (baseline)

3. Robust Evaluation:
   - 5-Fold Stratified Cross-Validation
   - Threshold optimization for F1 score
   - Weighted ensemble based on CV performance

4. Ensemble Predictions:
   - Weighted combination of all models
   - Blend with simple average for robustness
""")